# 118. 共享单车需求预测项目

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 33 / 34 步：把完整流程迁移到真实项目**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 物流延期风险预测项目  →  **本章任务：** 共享单车需求预测项目  →  **下一步：** 银行营销响应预测项目
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

运营共享单车的公司每天都在纠结同一件事——某个时段、某个天气条件下到
底会有多少人骑车。



## 本章目标

学完本章，你将能够：

- **理解**：理解「共享单车需求预测项目」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「共享单车需求预测项目」的关键输出指标。
- **迁移**：能把「共享单车需求预测项目」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 118.1 数据字典

**背景引入**：运营共享单车的公司每天都在纠结同一件事——某个时段、某个天气条件下到
底会有多少人骑车。把历史的小时骑行记录和气温、湿度、风速、是否工作日这些信息放在
一起，就能训练一个模型去预测未来的需求，从而提前调度车辆、避免高峰期无车可骑。这
一章就用一份公开的共享单车小时数据，从数据检查一直做到模型评价。

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| timestamp | 日期与小时 | 排序和切分依据 |
| workingday/weathersit | 工作日/天气 | 已知场景特征 |
| temp/hum/windspeed | 气象变量 | 归一化连续变量 |
| casual/registered | 需求组成 | 直接构成目标，禁止作为特征 |
| cnt | 总租赁量 | 回归目标 |

## 118.2 数据质量检查清单

- 时间重复、排序与缺失小时
- cnt是否等于casual加registered
- 缺失值和变量范围
- 训练时间严格早于测试
- 目标组成字段泄漏（打个比方：`cnt`（总租量）就是`casual`+`registered`拼出来的，拿拼它的零件去预测它，等于先知道结果再猜一遍，必然“准”但作弊。）
- 总体误差之外的时段差异


## 118.3 项目任务

1. 明确小时预测目标和预测提前量
2. 审计时间索引与目标构成
3. 识别并排除泄漏字段
4. 探索工作日、小时和天气差异
5. 构造小时与月份周期特征
6. 按时间划分训练、验证和测试
7. 建立季节基线并比较两个模型
8. 报告MAE、RMSE、R²与峰值识别
9. 分析残差切片和高误差时段
10. 解释特征重要性并总结局限


## 118.4 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 时间索引与数据质量审计 | `pd.read_csv()`、`pd.to_timedelta()`、`df.sort_values()`、`pd.Series()` | 组合日期和小时形成真正的时间主键，并检查排序、重复和缺失。 | 时间重复、排序与缺失小时 |
| 2. 目标构成与泄漏检查 | `.sum()`、`.describe()`、`.round()`、`df[['cnt','casual','registered']` | casual和registered相加就是cnt，若作为特征会让模型提前看到答案。 | cnt是否等于casual加registered |
| 3. 探索小时、工作日与天气场景 | `df.groupby()`、`cnt.mean()`、`x.quantile()`、`weather_profile.round()` | 探索用于认识数据和设计误差切片，不把组间差异直接解释为因果作用。 | 缺失值和变量范围 |
| 4. 构造周期时间特征 | `np.sin()`、`np.cos()`、`.head()`、`df['hour_sin']` | 正余弦编码让23点和0点、12月和1月在特征空间中保持相邻。 | 训练时间严格早于测试 |
| 5. 时间划分与季节基线 | `train.groupby()`、`cnt.mean()`、`np.array()`、`baseline_profile.get()` | 训练、验证和测试严格按时间排列；工作日×小时均值构成直观基线。 | 目标组成字段泄漏 |
| 6. 比较随机森林与梯度提升 | `models.items()`、`model.fit()`、`rows.append()`、`model.predict()` | 在验证时段上比较两个非线性回归模型，最终测试时段保持独立。 | 总体误差之外的时段差异 |
| 7. 最终回归指标与峰值识别 | `dev.groupby()`、`cnt.mean()`、`np.array()`、`test_profile.get()` | 总体回归误差之外，再检查模型是否识别出真实高需求时段。 | 时间重复、排序与缺失小时 |
| 8. 构造残差并进行时间切片 | `errors.residual.abs()`、`errors.groupby()`、`absolute_error.mean()`、`absolute_error.agg()` | 总体MAE可能掩盖特定小时、月份或工作日场景中的系统性误差。 | cnt是否等于casual加registered |
| 9. 检查高误差案例 | `errors.nlargest()`、`errors.nsmallest()`、`largest_under.round()`、`largest_over.round()` | 查看最大正负残差，判断模型在节假日、异常天气或需求突变时如何失效。 | 缺失值和变量范围 |
| 10. 特征解释与模型局限 | `np.linspace()`、`pd.Series()`、`importance.head()`、`.sort_values()` | 置换重要性展示模型依赖的预测信号，并明确数据只能支持全网小时需求建模。 | 训练时间严格早于测试 |


## 118.5 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 118.6 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 118.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 118.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 118.9 时间索引与数据质量审计

组合日期和小时形成真正的时间主键，并检查排序、重复和缺失。


<!-- math-foundation:chapter-118 -->
### 数学推导｜回归误差与解释度

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜定义每个样本残差。** $e_i=y_i-\hat y_i$。

**第 2 步｜选择如何汇总误差。** $MAE$ 平均绝对距离；$RMSE$ 先平均平方再开方，因此大残差权重更高。

**第 3 步｜与均值基线比较。** 常数模型 $\hat y_i=\bar y$ 的平方误差和是 $SST=\sum_i(y_i-\bar y)^2$，候选模型为 $SSE=\sum_ie_i^2$，所以

$$
R^2=1-\frac{SSE}{SST}
$$

$SSE>SST$ 时 $R^2<0$，表示还不如直接预测均值。

**把上面的关系收束为本章计算式：**

$$
MAE=\frac{1}{n}\sum_i|y_i-\hat{y}_i|,\qquad RMSE=\sqrt{\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2},\qquad R^2=1-\frac{\sum_i(y_i-\hat{y}_i)^2}{\sum_i(y_i-\bar{y})^2}
$$

**符号解释：** MAE 保留原单位，RMSE 更惩罚大误差，$R^2$ 相对均值基线衡量解释度。

**代码对应：** 至少同时报告一个原单位误差和基线比较，并检查高误差样本。

**使用边界：** $R^2$ 可以为负；不同目标尺度的数据不能只凭 RMSE 横向比较。


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("/datasets/bike_sharing_hour.csv", parse_dates=["dteday"])
df["timestamp"] = df.dteday + pd.to_timedelta(df.hr, unit="h")
df = df.sort_values("timestamp").reset_index(drop=True)
audit = pd.Series(
    {
        "行数": len(df),
        "时间重复": df.timestamp.duplicated().sum(),
        "缺失值": df.isna().sum().sum(),
        "时间是否递增": df.timestamp.is_monotonic_increasing,
    }
)
print(audit.to_string())
print("时间范围:", df.timestamp.min(), "至", df.timestamp.max())


**练一练**：数据终于读进来了，先别急着建模。请基于刚加载到变量 `df` 的共享单车小时数据，
统计两个最基础的质量指标——时间列 `timestamp` 的重复数量和整张表的缺失值总数——并分别
存入变量 `dup_timestamps` 与 `total_missing`（数值即可）。


In [ ]:
# 请在下方填写代码：统计时间列 timestamp 的重复数与全表缺失值总数
# 可参考的写法：df.timestamp.duplicated().sum()、df.isna().sum().sum()


In [ ]:
# 完整答案：基于变量 df 统计时间列重复与缺失值
dup_timestamps = df.timestamp.duplicated().sum()
total_missing = df.isna().sum().sum()


## 118.10 目标构成与泄漏检查

casual和registered相加就是cnt，若作为特征会让模型提前看到答案。


In [ ]:
composition_error = (df.cnt != df.casual + df.registered).sum()
forbidden = ["casual", "registered", "cnt"]
print("目标构成错误:", composition_error)
print("禁止进入特征:", forbidden)
display(df[["cnt", "casual", "registered"]].describe().round(1))


## 118.11 探索小时、工作日与天气场景

探索用于认识数据和设计误差切片，不把组间差异直接解释为因果作用。


In [ ]:
hour_profile = df.groupby(["workingday", "hr"]).cnt.mean()
weather_profile = df.groupby("weathersit").agg(
    hours=("cnt", "size"),
    mean_demand=("cnt", "mean"),
    p90=("cnt", lambda x: x.quantile(0.9)),
)
print("工作日需求最高小时:\n", hour_profile.loc[1].nlargest(5).round(1))
print("非工作日需求最高小时:\n", hour_profile.loc[0].nlargest(5).round(1))
display(weather_profile.round(1))


## 118.12 构造周期时间特征

正余弦编码让23点和0点、12月和1月在特征空间中保持相邻。


In [ ]:
df["hour_sin"] = np.sin(2 * np.pi * df.hr / 24)
df["hour_cos"] = np.cos(2 * np.pi * df.hr / 24)
df["month_sin"] = np.sin(2 * np.pi * df.mnth / 12)
df["month_cos"] = np.cos(2 * np.pi * df.mnth / 12)
features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit",
    "temp",
    "atemp",
    "hum",
    "windspeed",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
]
_check_1 = bool(not set(features) & set(forbidden))
print(
    "自检 1：not set(features)&set(forbidden) ->",
    "通过" if _check_1 else "需要检查",
)
if not _check_1:
    print("建议：", "请回看输入、处理步骤和预期结果。")
print("特征数量:", len(features))
display(
    df[["hr", "hour_sin", "hour_cos", "mnth", "month_sin", "month_cos"]].head()
)


## 118.13 时间划分与季节基线

训练、验证和测试严格按时间排列；工作日×小时均值构成直观基线。


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train_end = int(len(df) * 0.64)
val_end = int(len(df) * 0.80)
train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]
baseline_profile = train.groupby(["workingday", "hr"]).cnt.mean()
val_baseline = np.array(
    [
        baseline_profile.get((w, h), train.cnt.mean())
        for w, h in zip(val.workingday, val.hr)
    ]
)
print("训练/验证/测试:", len(train), len(val), len(test))
print("训练截止:", train.timestamp.max(), "测试开始:", test.timestamp.min())
print(
    "验证集季节基线MAE:", round(mean_absolute_error(val.cnt, val_baseline), 1)
)


## 118.14 比较随机森林与梯度提升

在验证时段上比较两个非线性回归模型，最终测试时段保持独立。


In [ ]:
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)

models = {
    "随机森林": RandomForestRegressor(
        n_estimators=180, min_samples_leaf=3, n_jobs=-1, random_state=107
    ),
    "梯度提升": HistGradientBoostingRegressor(
        max_iter=220, max_leaf_nodes=20, l2_regularization=1, random_state=107
    ),
}
rows = []
for name, model in models.items():
    model.fit(train[features], train.cnt)
    rows.append(
        [name, mean_absolute_error(val.cnt, model.predict(val[features]))]
    )
validation = pd.DataFrame(
    rows, columns=["model", "validation_MAE"]
).sort_values("validation_MAE")
display(validation.round(1))
best_name = validation.iloc[0].model
dev = df.iloc[:val_end]
best_model = models[best_name].fit(dev[features], dev.cnt)
prediction = np.maximum(0, best_model.predict(test[features]))


## 118.15 最终回归指标与峰值识别

总体回归误差之外，再检查模型是否识别出真实高需求时段。


In [ ]:
test_profile = dev.groupby(["workingday", "hr"]).cnt.mean()
test_baseline = np.array(
    [
        test_profile.get((w, h), dev.cnt.mean())
        for w, h in zip(test.workingday, test.hr)
    ]
)
metrics = pd.Series(
    {
        "MAE": mean_absolute_error(test.cnt, prediction),
        "RMSE": mean_squared_error(test.cnt, prediction) ** 0.5,
        "R2": r2_score(test.cnt, prediction),
        "Baseline_MAE": mean_absolute_error(test.cnt, test_baseline),
    }
)
peak_threshold = dev.cnt.quantile(0.9)
actual_peak = test.cnt >= peak_threshold
predicted_peak = prediction >= peak_threshold
peak_precision = ((actual_peak) & (predicted_peak)).sum() / max(
    predicted_peak.sum(), 1
)
peak_recall = ((actual_peak) & (predicted_peak)).sum() / max(
    actual_peak.sum(), 1
)
print("最佳模型:", best_name)
print(metrics.round(2).to_string())
print(
    "高需求阈值:",
    round(peak_threshold, 1),
    "峰值Precision:",
    round(peak_precision, 3),
    "峰值Recall:",
    round(peak_recall, 3),
)


## 118.16 构造残差并进行时间切片

总体MAE可能掩盖特定小时、月份或工作日场景中的系统性误差。


In [ ]:
errors = test[
    ["timestamp", "hr", "mnth", "workingday", "weathersit", "cnt"]
].copy()
errors["prediction"] = prediction
errors["residual"] = errors.cnt - errors.prediction
errors["absolute_error"] = errors.residual.abs()
hour_error = errors.groupby("hr").absolute_error.mean().nlargest(6)
month_error = errors.groupby("mnth").absolute_error.mean().nlargest(4)
workday_error = errors.groupby("workingday").absolute_error.agg(
    ["count", "mean", "median"]
)
print("误差最高小时:\n", hour_error.round(1))
print("误差最高月份:\n", month_error.round(1))
print("工作日切片:\n", workday_error.round(1))


## 118.17 检查高误差案例

查看最大正负残差，判断模型在节假日、异常天气或需求突变时如何失效。


In [ ]:
largest_under = errors.nlargest(6, "residual")[
    ["timestamp", "cnt", "prediction", "residual", "workingday", "weathersit"]
]
largest_over = errors.nsmallest(6, "residual")[
    ["timestamp", "cnt", "prediction", "residual", "workingday", "weathersit"]
]
print("明显低估案例:")
display(largest_under.round(1))
print("明显高估案例:")
display(largest_over.round(1))


## 118.18 特征解释与模型局限

置换重要性展示模型依赖的预测信号，并明确数据只能支持全网小时需求建模。


In [ ]:
from sklearn.inspection import permutation_importance

sample_n = min(3000, len(test))
sample_idx = np.linspace(0, len(test) - 1, sample_n, dtype=int)
permutation = permutation_importance(
    best_model,
    test.iloc[sample_idx][features],
    test.iloc[sample_idx].cnt,
    n_repeats=3,
    scoring="neg_mean_absolute_error",
    random_state=107,
    n_jobs=-1,
)
importance = pd.Series(
    permutation.importances_mean, index=features
).sort_values(ascending=False)
print("置换重要性前10:\n", importance.head(10).round(2))
print(
    "局限: 数据没有站点库存、OD流向和未来天气预报，不能据此完成站点级调度或因果解释。"
)


## 118.19 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 118.19.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 118.19.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 118.20 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 118.20.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 118.21 易错点提醒

**易错点 1**：预测目标是"未来某小时的 cnt"，特征只能取该小时之前的数据；把目标小时自身的特征当特征就是泄漏。

**易错点 2**：随机切分训练/测试会破坏时间顺序，需求预测必须按时间先后切分（留出未来时段）。

**易错点 3**：天气变量若用"当天实测天气"建模，预测未来时拿不到，应使用预报值或把天气排除出特征。

**易错点 4**：hr 是类别型小时编号，直接当数值进模型会假设"23 点与 0 点相邻"，建议按业务时段分桶。

**易错点 5**：cnt 波动大（雨天几乎归零），评估用 RMSE/MAE 时先看分组误差，别只报一个总体数字。


## 118.22 结论与表达

- 时间数据不能随机打乱后评价未来表现
- 目标组成字段是最直接的数据泄漏
- 季节基线能判断复杂模型是否真正提供增益
- 总体误差必须结合峰值和时间切片理解


## 118.23 项目验收清单

- 完成时间索引和目标构成审计
- 排除casual与registered
- 使用严格时间三段划分
- 比较季节基线和两个模型
- 报告回归指标、峰值指标和误差切片
- 完成高误差案例与置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 118.24 小结

使用 UCI Bike Sharing 小时数据，按照时间序列回归的教学流程预测共享单车小时需求。


### 118.24.1 你已经完成

- 审计时间索引和目标构成
- 识别目标组成字段造成的直接泄漏
- 构造周期时间特征
- 使用时间顺序划分和季节基线
- 比较随机森林与梯度提升
- 通过残差切片和特征重要性理解模型


### 118.24.2 质量与结论提醒

- 时间重复、排序与缺失小时
- cnt是否等于casual加registered
- 缺失值和变量范围
- 时间数据不能随机打乱后评价未来表现
- 目标组成字段是最直接的数据泄漏
- 季节基线能判断复杂模型是否真正提供增益
- 总体误差必须结合峰值和时间切片理解


### 118.24.3 学习检查

- [ ] 完成时间索引和目标构成审计
- [ ] 排除casual与registered
- [ ] 使用严格时间三段划分
- [ ] 比较季节基线和两个模型
- [ ] 报告回归指标、峰值指标和误差切片
- [ ] 完成高误差案例与置换重要性分析


### 118.24.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
